In [1]:
from pathlib import Path
import os
import sys
import platform
import shutil
import subprocess
import importlib
import importlib.util
import importlib.metadata as metadata
import json
import time
import gc

ROOT = Path.cwd().resolve()

print("Notebook 当前目录：")
print(ROOT)

assert (ROOT / "pyproject.toml").is_file(), (
    "当前目录没有 pyproject.toml。\n"
    "请关闭 Notebook，将它移动到 Swift 源码根目录后重新打开。"
)

assert (ROOT / "src" / "swift").is_dir(), (
    "没有找到 src/swift。\n"
    "说明 Notebook 不在正确的 Swift 源码根目录。"
)

print("\n源码根目录检查通过。")
print("pyproject.toml：", ROOT / "pyproject.toml")
print("Swift 包目录：", ROOT / "src" / "swift")

Notebook 当前目录：
/root/private_data/zc/swift-main

源码根目录检查通过。
pyproject.toml： /root/private_data/zc/swift-main/pyproject.toml
Swift 包目录： /root/private_data/zc/swift-main/src/swift


In [2]:
import torch

def get_module_version(module_name, distribution_name=None):
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", None)
        if version is not None:
            return str(version)
    except Exception:
        pass

    try:
        return metadata.version(distribution_name or module_name)
    except Exception:
        return "未知"

def read_total_memory_gb():
    meminfo = Path("/proc/meminfo")
    if not meminfo.exists():
        return None

    for line in meminfo.read_text().splitlines():
        if line.startswith("MemTotal:"):
            kb = int(line.split()[1])
            return kb / 1024**2
    return None

print("=" * 70)
print("基础运行环境")
print("=" * 70)
print("Python 可执行文件：", sys.executable)
print("Python 版本：", sys.version.replace("\n", " "))
print("操作系统：", platform.platform())
print("机器架构：", platform.machine())
print("主机名：", platform.node())
print("CPU 信息：", platform.processor() or "未提供")

ram_gb = read_total_memory_gb()
if ram_gb is not None:
    print(f"系统内存：{ram_gb:.2f} GB")

disk = shutil.disk_usage(ROOT)
print(f"当前磁盘总量：{disk.total / 1024**3:.2f} GB")
print(f"当前磁盘可用：{disk.free / 1024**3:.2f} GB")

print("\nPyTorch 信息")
print("-" * 70)
print("torch 版本：", torch.__version__)
print("torch CUDA 编译版本：", torch.version.cuda)
print("torch CUDA 是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA 设备数量：", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(
            f"设备 {i}：{props.name}，"
            f"显存 {props.total_memory / 1024**3:.2f} GB"
        )

# 部分非 NVIDIA 后端会把设备接口注册到 torch.npu、torch.xpu、torch.musa 等
for backend_name in ["npu", "xpu", "musa", "mlu"]:
    backend = getattr(torch, backend_name, None)
    if backend is None:
        continue

    is_available = getattr(backend, "is_available", None)
    if callable(is_available):
        try:
            available = is_available()
        except Exception:
            available = False

        print(f"torch.{backend_name} 是否可用：", available)

        if available:
            count_fn = getattr(backend, "device_count", None)
            name_fn = getattr(backend, "get_device_name", None)

            if callable(count_fn):
                count = count_fn()
                print(f"{backend_name} 设备数量：", count)

                if callable(name_fn):
                    for i in range(count):
                        try:
                            print(f"设备 {i}：", name_fn(i))
                        except Exception:
                            pass

print("\n可能的镜像信息环境变量")
print("-" * 70)

image_keys = [
    "FLAGOS_IMAGE",
    "IMAGE_NAME",
    "JUPYTER_IMAGE_SPEC",
    "JUPYTERHUB_IMAGE",
    "CONDA_DEFAULT_ENV",
    "HOSTNAME",
]

IMAGE_INFO = {}

for key in image_keys:
    value = os.environ.get(key)
    if value:
        IMAGE_INFO[key] = value
        print(f"{key}={value}")

if not IMAGE_INFO:
    print("Notebook 内没有暴露明确镜像名。")
    print("请从 FlagOS/SCNET 创建 Notebook 的页面手动记录：")
    print("1. 所属组")
    print("2. 镜像完整名称")
    print("3. 镜像版本或标签")

基础运行环境
Python 可执行文件： /usr/bin/python3
Python 版本： 3.10.12 (main, Feb  4 2025, 14:57:36) [GCC 11.4.0]
操作系统： Linux-3.10.0-957.el7.x86_64-x86_64-with-glibc2.39
机器架构： x86_64
主机名： crdnotebook-2080100196699160577-onesciencetes859-82845
CPU 信息： x86_64
系统内存：1007.43 GB
当前磁盘总量：2000.00 GB
当前磁盘可用：1781.25 GB

PyTorch 信息
----------------------------------------------------------------------
torch 版本： 2.4.1
torch CUDA 编译版本： None
torch CUDA 是否可用： True
CUDA 设备数量： 1
设备 0：K100_AI，显存 63.98 GB
torch.xpu 是否可用： False

可能的镜像信息环境变量
----------------------------------------------------------------------
HOSTNAME=crdnotebook-2080100196699160577-onesciencetes859-82845


In [3]:
CORE_MODULES = ["torch", "triton", "flag_gems"]

core_results = {}

for name in CORE_MODULES:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", "未提供 __version__")
        core_results[name] = {
            "status": "成功",
            "version": str(version),
        }
        print(f"[成功] import {name}")
        print(f"       版本：{version}")
    except Exception as exc:
        core_results[name] = {
            "status": "失败",
            "error": repr(exc),
        }
        print(f"[失败] import {name}")
        print(f"       错误：{exc!r}")

failed_core = [
    name for name, result in core_results.items()
    if result["status"] != "成功"
]

assert not failed_core, (
    f"FlagOS 镜像缺少核心组件：{failed_core}\n"
    "不要在这里直接 pip install 普通 torch 或普通 triton。\n"
    "请重新选择正确的 FlagOS/FlagGems 镜像。"
)

import triton
import flag_gems

print("\nFlagGems 设备：", getattr(flag_gems, "device", "未提供"))
print("\n核心环境检查全部通过。")
print("注意：这里只导入了 flag_gems，还没有启用 FlagGems 算子。")

[成功] import torch
       版本：2.4.1
[成功] import triton
       版本：3.0.0
[成功] import flag_gems
       版本：5.0.0

FlagGems 设备： cuda

核心环境检查全部通过。
注意：这里只导入了 flag_gems，还没有启用 FlagGems 算子。


In [4]:
assert sys.version_info >= (3, 10), (
    f"Swift 要求 Python >= 3.10，当前版本是 {sys.version}"
)

install_swift_command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
    "--no-deps",
    "-e",
    str(ROOT),
]

print("即将执行：")
print(" ".join(install_swift_command))

subprocess.check_call(install_swift_command)

# 保证当前 Notebook Kernel 不重启也能立即看到源码
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("\nSwift 源码安装完成。")

即将执行：
/usr/bin/python3 -m pip install --no-cache-dir --no-deps -e /root/private_data/zc/swift-main


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Obtaining file:///root/private_data/zc/swift-main
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for swift (pyproject.toml): started
  Building editable for swift (pyproject.toml): finished with status 'done'
  Created wheel for swift: filename=swift-0.0-py3-none-any.whl size=3905 sha256=62626d919897a43fc04267d9b53690e344b0d3d45683fcd8284e16579b0ab390
  Stored i

In [5]:
from packaging.version import Version

RUNTIME_REQUIREMENTS = {
    # 导入模块名: pip 安装名
    "numpy": "numpy",
    "h5py": "h5py>=3.13.0",
    "omegaconf": "omegaconf",
    "hydra": "hydra-core",
    "einops": "einops",
    "tqdm": "tqdm",
    "requests": "requests",
}

INSTALL_NOW = []

for module_name, pip_spec in RUNTIME_REQUIREMENTS.items():
    if importlib.util.find_spec(module_name) is None:
        INSTALL_NOW.append(pip_spec)
        print(f"[缺失] {module_name} -> 准备安装 {pip_spec}")
    else:
        print(f"[已有] {module_name}")

# Swift 源码要求 h5py >= 3.13.0
if importlib.util.find_spec("h5py") is not None:
    try:
        current_h5py = Version(metadata.version("h5py"))
        if current_h5py < Version("3.13.0"):
            if "h5py>=3.13.0" not in INSTALL_NOW:
                INSTALL_NOW.append("h5py>=3.13.0")
            print(
                f"[版本偏低] h5py={current_h5py}，"
                "源码要求至少为 3.13.0"
            )
    except Exception as exc:
        print("无法判断 h5py 版本：", exc)

# 去重，同时保持顺序
INSTALL_NOW = list(dict.fromkeys(INSTALL_NOW))
INSTALLED_THIS_RUN = INSTALL_NOW.copy()

print("\n本次需要安装：", INSTALL_NOW if INSTALL_NOW else "无")

if INSTALL_NOW:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *INSTALL_NOW,
    ]

    print("\n执行：")
    print(" ".join(command))
    subprocess.check_call(command)
else:
    print("不需要安装额外运行包。")

print("\n重新检查导入：")

for module_name in RUNTIME_REQUIREMENTS:
    module = importlib.import_module(module_name)
    print(
        f"[成功] {module_name}:",
        getattr(module, "__version__", "未提供版本号")
    )

[已有] numpy
[已有] h5py
[已有] omegaconf
[已有] hydra
[已有] einops
[已有] tqdm
[已有] requests

本次需要安装： 无
不需要安装额外运行包。

重新检查导入：
[成功] numpy: 1.26.3
[成功] h5py: 3.16.0
[成功] omegaconf: 2.3.0
[成功] hydra: 1.3.2
[成功] einops: 0.8.1
[成功] tqdm: 4.66.6
[成功] requests: 2.34.2


In [6]:
MODEL_OPTIONS = {
    "swift": {
        "display_name": "Swift",
        "base": "weights/swift/020000",
        "checkpoint": "checkpoint-020000.pt",
    },
    "swift-b": {
        "display_name": "Swift-B",
        "base": "weights/swift/015000",
        "checkpoint": "checkpoint-015000.pt",
    },
}

# 默认验证 README 中的主 Swift 模型
MODEL_CHOICE = "swift"

assert MODEL_CHOICE in MODEL_OPTIONS

MODEL_INFO = MODEL_OPTIONS[MODEL_CHOICE]
MODEL_BASE = MODEL_INFO["base"]
CHECKPOINT_NAME = MODEL_INFO["checkpoint"]

CONFIG_PATH = ROOT / MODEL_BASE / ".hydra" / "config.yaml"
CHECKPOINT_PATH = ROOT / MODEL_BASE / "checkpoints" / CHECKPOINT_NAME
DATA_ROOT = ROOT / "sample-data"

free_gb = shutil.disk_usage(ROOT).free / 1024**3

print("选择的模型：", MODEL_INFO["display_name"])
print("配置文件目标位置：", CONFIG_PATH)
print("Checkpoint 目标位置：", CHECKPOINT_PATH)
print("样例数据目标位置：", DATA_ROOT)
print(f"当前磁盘剩余：{free_gb:.2f} GB")

if MODEL_CHOICE == "swift" and free_gb < 6:
    print("\n警告：磁盘空间可能不足。")
    print('可以把 MODEL_CHOICE 改为 "swift-b"，然后重新执行本单元。')

选择的模型： Swift
配置文件目标位置： /root/private_data/zc/swift-main/weights/swift/020000/.hydra/config.yaml
Checkpoint 目标位置： /root/private_data/zc/swift-main/weights/swift/020000/checkpoints/checkpoint-020000.pt
样例数据目标位置： /root/private_data/zc/swift-main/sample-data
当前磁盘剩余：1781.25 GB


In [7]:
from urllib.parse import quote
import requests
from tqdm.auto import tqdm

HF_REPO = "stockeh/swift-era5-1.4"

REQUIRED_REMOTE_FILES = [
    f"{MODEL_BASE}/.hydra/config.yaml",
    f"{MODEL_BASE}/checkpoints/{CHECKPOINT_NAME}",
    "sample-data/normalize_mean.npz",
    "sample-data/normalize_std.npz",
    "sample-data/normalize_diff_std_6.npz",
    "sample-data/test/2020_0937.h5",
    "sample-data/test/2020_0938.h5",
]

def download_hf_file(relative_path: str) -> Path:
    """
    从 Hugging Face 直接下载到源码目录。
    支持断点续传。
    下载中使用 .part，完成后自动改为正式文件名。
    """
    destination = ROOT / relative_path

    if destination.exists() and destination.stat().st_size > 0:
        print(
            f"[跳过] {relative_path} "
            f"({destination.stat().st_size / 1024**2:.2f} MB)"
        )
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)

    temporary = Path(str(destination) + ".part")
    resume_size = temporary.stat().st_size if temporary.exists() else 0

    encoded_path = quote(relative_path, safe="/")
    url = (
        f"https://huggingface.co/{HF_REPO}/resolve/main/"
        f"{encoded_path}?download=true"
    )

    def make_request(offset: int):
        headers = {}
        if offset > 0:
            headers["Range"] = f"bytes={offset}-"

        return requests.get(
            url,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=(30, 600),
        )

    response = make_request(resume_size)

    # 服务端不接受 Range 时，从头下载
    if resume_size > 0 and response.status_code != 206:
        response.close()
        temporary.unlink(missing_ok=True)
        resume_size = 0
        response = make_request(0)

    response.raise_for_status()

    remaining = int(response.headers.get("content-length") or 0)
    total = resume_size + remaining if remaining > 0 else None
    mode = "ab" if resume_size > 0 else "wb"

    with response:
        with temporary.open(mode) as output:
            with tqdm(
                total=total,
                initial=resume_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=destination.name,
            ) as progress:
                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if not chunk:
                        continue

                    output.write(chunk)
                    progress.update(len(chunk))

    temporary.replace(destination)

    print(
        f"[完成] {relative_path} "
        f"({destination.stat().st_size / 1024**2:.2f} MB)"
    )

    return destination

print("开始检查或下载必需文件：\n")

for relative_path in REQUIRED_REMOTE_FILES:
    download_hf_file(relative_path)

print("\n全部必需文件处理完成。")

开始检查或下载必需文件：

[跳过] weights/swift/020000/.hydra/config.yaml (0.00 MB)
[跳过] weights/swift/020000/checkpoints/checkpoint-020000.pt (3448.46 MB)
[跳过] sample-data/normalize_mean.npz (0.03 MB)
[跳过] sample-data/normalize_std.npz (0.03 MB)
[跳过] sample-data/normalize_diff_std_6.npz (0.03 MB)
[跳过] sample-data/test/2020_0937.h5 (16.30 MB)
[跳过] sample-data/test/2020_0938.h5 (16.30 MB)

全部必需文件处理完成。


In [8]:
print("文件完整性初步检查：\n")

for relative_path in REQUIRED_REMOTE_FILES:
    path = ROOT / relative_path

    assert path.exists(), f"文件不存在：{path}"
    assert path.stat().st_size > 1000, f"文件过小，可能下载失败：{path}"

    print(
        f"[存在] {relative_path:<75} "
        f"{path.stat().st_size / 1024**2:>10.2f} MB"
    )

assert CHECKPOINT_PATH.stat().st_size > 1024**3, (
    "Checkpoint 小于 1 GB，可能没有下载完整。"
)

print("\n文件初步检查通过。")

文件完整性初步检查：

[存在] weights/swift/020000/.hydra/config.yaml                                           0.00 MB
[存在] weights/swift/020000/checkpoints/checkpoint-020000.pt                          3448.46 MB
[存在] sample-data/normalize_mean.npz                                                    0.03 MB
[存在] sample-data/normalize_std.npz                                                     0.03 MB
[存在] sample-data/normalize_diff_std_6.npz                                              0.03 MB
[存在] sample-data/test/2020_0937.h5                                                    16.30 MB
[存在] sample-data/test/2020_0938.h5                                                    16.30 MB

文件初步检查通过。


In [9]:
import numpy as np
from omegaconf import OmegaConf
from hydra.utils import instantiate

np.random.seed(1118)
torch.manual_seed(1118)

cfg = OmegaConf.load(CONFIG_PATH)

print("配置中原始数据路径：")
print(cfg.data.dataset.root)

# 只修改内存中的 cfg，不改写 config.yaml
OmegaConf.update(
    cfg,
    "data.dataset.root",
    str(DATA_ROOT),
    merge=False,
)

# 当前只有两个连续样本，只进行 6 小时一步预测
OmegaConf.update(
    cfg,
    "data.dataset.intervals",
    [6],
    merge=False,
)

# Notebook 单进程读取，避免多进程 DataLoader 干扰
OmegaConf.update(
    cfg,
    "data.data_workers",
    0,
    merge=False,
)

print("\n内存中修改后的数据路径：")
print(cfg.data.dataset.root)

print("测试时间间隔：", list(cfg.data.dataset.intervals))
print("模型类：", cfg.model._target_)
print("预处理/预条件类：", cfg.precond._target_)
print("模型维度 dim：", cfg.model.dim)
print("模型深度 depth：", cfg.model.depth)
print("注意：磁盘上的 config.yaml 没有被修改。")

配置中原始数据路径：
/lus/flare/projects/SAFS/jstock/data/wb2/1.40625deg_1_step_6hr_h5df

内存中修改后的数据路径：
/root/private_data/zc/swift-main/sample-data
测试时间间隔： [6]
模型类： swift.models.swinv2.SwinV2
预处理/预条件类： swift.models.precond.PassPrecond
模型维度 dim： 1056
模型深度 depth： 12
注意：磁盘上的 config.yaml 没有被修改。


In [10]:
dataset = instantiate(
    cfg.data.dataset,
    split="test",
    _convert_="object",
)

print("数据根目录：", dataset.root)
print("找到的 H5 文件：")

for file_path in dataset.files:
    print(" -", Path(file_path).name)

print("\n可用数据集长度：", len(dataset))
print("天气变量数量：", len(dataset.variables))
print("强迫变量数量：", len(dataset.forcings))
print("目标通道数量：", dataset.n_target_channels)
print("条件通道数量：", dataset.n_condition_channels)
print("空间大小：", dataset.img_resolution)
print("Residual 模式：", dataset.residual)

assert len(dataset) >= 1, (
    "数据集长度为 0。\n"
    "请确认：\n"
    "1. 两个 H5 文件都下载成功；\n"
    "2. cfg.data.dataset.intervals 已设置为 [6]。"
)

# 明确指定：
# idx=0，offset=1，delta=6 小时
(x, target_residual), (sample_index, auxiliary) = dataset[(0, 1, 6)]

print("\n单个输入张量形状：", tuple(x.shape))
print("单个目标张量形状：", tuple(target_residual.shape))
print("样本编号：", sample_index)
print("辅助时间条件：", auxiliary.item())

assert torch.isfinite(x).all(), "输入中出现 NaN 或 Inf。"
assert torch.isfinite(target_residual).all(), "目标中出现 NaN 或 Inf。"

assert x.shape[-2:] == torch.Size([128, 256]), (
    f"空间尺寸异常：{tuple(x.shape[-2:])}"
)

print("\n样例数据加载成功，输入和目标均为有限数值。")

数据根目录： /root/private_data/zc/swift-main/sample-data
找到的 H5 文件：
 - 2020_0937.h5
 - 2020_0938.h5

可用数据集长度： 1
天气变量数量： 69
强迫变量数量： 3
目标通道数量： 69
条件通道数量： 72
空间大小： (128, 256)
Residual 模式： True

单个输入张量形状： (72, 128, 256)
单个目标张量形状： (69, 128, 256)
样本编号： 0
辅助时间条件： 0.6000000238418579

样例数据加载成功，输入和目标均为有限数值。


In [11]:
# FlagGems 官方示例使用 flag_gems.device 作为设备
raw_device = getattr(flag_gems, "device", None)

assert raw_device is not None, "flag_gems 没有提供 device。"

DEVICE = (
    raw_device
    if isinstance(raw_device, torch.device)
    else torch.device(raw_device)
)

print("即将使用的设备：", DEVICE)

# 尽可能匹配原始配置
if hasattr(torch, "set_float32_matmul_precision"):
    precision = str(
        getattr(cfg.system.torch, "set_float32_matmul_precision", "high")
    )
    torch.set_float32_matmul_precision(precision)
    print("float32 matmul precision：", precision)

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = bool(
        getattr(cfg.system.torch, "allow_tf32", True)
    )
    torch.backends.cudnn.benchmark = bool(
        getattr(cfg.system.torch, "benchmark", True)
    )

print("\n开始在 CPU 上构造模型……")

net = instantiate(
    cfg.precond,
    model_config=cfg.model,
    img_resolution=dataset.img_resolution,
    img_channels=dataset.n_target_channels,
    condition_channels=dataset.n_condition_channels,
    sigma_max=float("inf"),
    _recursive_=False,
    _convert_="object",
)

parameter_count = sum(parameter.numel() for parameter in net.parameters())

print(f"模型参数量：{parameter_count:,}")
print(f"约为：{parameter_count / 1e6:.2f} M 参数")

def load_checkpoint_compatibly(path: Path):
    """
    优先使用 weights_only 和 mmap。
    对较旧或厂商定制 PyTorch 提供兼容回退。
    """
    attempts = [
        {"weights_only": True, "mmap": True},
        {"weights_only": True},
        {},
    ]

    last_error = None

    for kwargs in attempts:
        try:
            print("尝试 torch.load 参数：", kwargs)
            return torch.load(
                path,
                map_location="cpu",
                **kwargs,
            )
        except (TypeError, RuntimeError) as exc:
            last_error = exc
            print("本次加载方式不可用：", repr(exc))

    raise RuntimeError(
        f"所有 checkpoint 加载方式均失败。最后错误：{last_error!r}"
    )

print("\n开始读取 checkpoint：")
print(CHECKPOINT_PATH)

state = load_checkpoint_compatibly(CHECKPOINT_PATH)

assert isinstance(state, dict), "Checkpoint 顶层不是字典。"
assert "ema" in state, "Checkpoint 中没有找到 ema 权重。"

print("Checkpoint 顶层键：", list(state.keys()))

load_result = net.load_state_dict(state["ema"], strict=True)

print("权重加载结果：", load_result)

del state
gc.collect()

print("\n将模型移动到加速设备……")

net = net.to(DEVICE)
net.eval()

print("模型当前设备：", next(net.parameters()).device)
print("模型 checkpoint 加载完成。")

即将使用的设备： cuda
float32 matmul precision： high

开始在 CPU 上构造模型……
模型参数量：225,980,976
约为：225.98 M 参数

开始读取 checkpoint：
/root/private_data/zc/swift-main/weights/swift/020000/checkpoints/checkpoint-020000.pt
尝试 torch.load 参数： {'weights_only': True, 'mmap': True}
Checkpoint 顶层键： ['ema', 'net', 'optimizer', 'scaler']
权重加载结果： <All keys matched successfully>

将模型移动到加速设备……
模型当前设备： cuda:0
模型 checkpoint 加载完成。


In [12]:
from swift.generating.factory import sampler_factory

# x 已经包含：
# [天气状态变量, 外部强迫变量]
condition = x.unsqueeze(0).to(DEVICE)

print("模型条件输入形状：", tuple(condition.shape))
print("模型要求的条件通道：", dataset.n_condition_channels)

assert condition.shape[1] == dataset.n_condition_channels, (
    "输入条件通道数与模型配置不一致。"
)

solver_kwargs = {
    "num_steps": 1,
    "sigma_min": 0.02,
    "sigma_max": 200.0,
    "auxiliary": 0.6,  # 6 小时 / 10
}

sampler = sampler_factory(
    "scm",
    net,
    **solver_kwargs,
)

def synchronize_device():
    backend = getattr(torch, DEVICE.type, None)
    synchronize = getattr(backend, "synchronize", None)

    if callable(synchronize):
        synchronize()

def reset_peak_memory():
    backend = getattr(torch, DEVICE.type, None)
    reset_fn = getattr(backend, "reset_peak_memory_stats", None)

    if callable(reset_fn):
        try:
            reset_fn()
        except Exception:
            pass

def get_peak_memory_gb():
    backend = getattr(torch, DEVICE.type, None)
    memory_fn = getattr(backend, "max_memory_allocated", None)

    if callable(memory_fn):
        try:
            return memory_fn() / 1024**3
        except Exception:
            return None

    return None

def run_one_step(seed: int = 0):
    """
    用固定随机种子完成一次 SCM 一步推理。
    返回 CPU float32 输出、运行时间和峰值显存。
    """
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    reset_peak_memory()
    synchronize_device()

    start = time.perf_counter()

    with torch.inference_mode():
        output = sampler(
            condition.clone(),
            generator=generator,
        )

    synchronize_device()

    elapsed = time.perf_counter() - start
    peak_memory = get_peak_memory_gb()

    output_cpu = output.detach().float().cpu()

    return output_cpu, elapsed, peak_memory

print("一步推理函数构造完成。")
print("此时 FlagGems 尚未启用。")

模型条件输入形状： (1, 72, 128, 256)
模型要求的条件通道： 72
一步推理函数构造完成。
此时 FlagGems 尚未启用。


In [13]:
print("开始原生 PyTorch 基线推理……")

baseline_output, baseline_time, baseline_peak_memory = run_one_step(seed=0)

print("\n原生 PyTorch 推理完成。")
print("输出形状：", tuple(baseline_output.shape))
print("运行时间：", f"{baseline_time:.6f} 秒")
print("输出最小值：", baseline_output.min().item())
print("输出最大值：", baseline_output.max().item())
print("输出平均值：", baseline_output.mean().item())
print("输出标准差：", baseline_output.std().item())
print("全部为有限数值：", bool(torch.isfinite(baseline_output).all()))

if baseline_peak_memory is not None:
    print("峰值设备内存：", f"{baseline_peak_memory:.3f} GB")

assert torch.isfinite(baseline_output).all(), (
    "原生 PyTorch 输出存在 NaN 或 Inf，不能继续测试 FlagGems。"
)

开始原生 PyTorch 基线推理……


/root/private_data/zc/swift-main/src/swift/models/swinv2.py:130: UserWarning: 1Torch was not compiled with memory efficient attention. (Triggered internally at /home/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:627.)
  x = F.scaled_dot_product_attention(q, k, v, scale=1.0)



原生 PyTorch 推理完成。
输出形状： (1, 69, 128, 256)
运行时间： 15.709348 秒
输出最小值： -17.961395263671875
输出最大值： 21.255802154541016
输出平均值： 0.001595409819856286
输出标准差： 1.0478705167770386
全部为有限数值： True
峰值设备内存： 1.399 GB


In [14]:
variable_count = len(dataset.variables)

# 输入中的天气状态部分，不包括 forcing
initial_state_std = x[:variable_count].unsqueeze(0).cpu()

initial_state_physical = dataset.unstandardize_x(
    initial_state_std,
    delta=6,
)

true_residual_physical = dataset.unstandardize_t(
    target_residual.unsqueeze(0).cpu(),
    delta=6,
)

baseline_residual_physical = dataset.unstandardize_t(
    baseline_output,
    delta=6,
)

true_next_state = initial_state_physical + true_residual_physical
baseline_next_state = (
    initial_state_physical + baseline_residual_physical
)

t2m_index = dataset.variables.index("2m_temperature")

baseline_t2m_rmse = torch.sqrt(
    torch.mean(
        (
            baseline_next_state[:, t2m_index]
            - true_next_state[:, t2m_index]
        ) ** 2
    )
).item()

print("原生 PyTorch 2m_temperature 单样本 RMSE：")
print(baseline_t2m_rmse)

原生 PyTorch 2m_temperature 单样本 RMSE：
0.6926568150520325


In [15]:
from pathlib import Path
import flag_gems

GEMS_LOG_PATH = ROOT / "gems_debug.log"

# 删除上一次失败运行留下的旧日志
if GEMS_LOG_PATH.exists():
    GEMS_LOG_PATH.unlink()
    print("已删除旧日志：", GEMS_LOG_PATH)

# 本次实际禁用并回退到 PyTorch 的算子
UNUSED_OPS = [
    "batch_norm",
    "batch_norm_backward",

    # 当前 GPU 的共享内存上限为 65536 Bytes，
    # FlagGems mm 内核要求 131072 Bytes，因此禁用
    "mm",
]

print("准备启用 FlagGems")
print("回退到 PyTorch 的算子：", UNUSED_OPS)

flag_gems.enable(
    unused=UNUSED_OPS,
    record=True,
    path=str(GEMS_LOG_PATH),
    once=True,
)

print("\nFlagGems 已启用")
print("日志路径：", GEMS_LOG_PATH)
print("mm 将使用 PyTorch 原生实现")

已删除旧日志： /root/private_data/zc/swift-main/gems_debug.log
准备启用 FlagGems
回退到 PyTorch 的算子： ['batch_norm', 'batch_norm_backward', 'mm']

FlagGems 已启用
日志路径： /root/private_data/zc/swift-main/gems_debug.log
mm 将使用 PyTorch 原生实现


/usr/local/lib/python3.10/dist-packages/torch/library.py:255: UserWarning: Warning only once for all operators,  other operators may also be overrided.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::_flash_attention_forward(Tensor query, Tensor key, Tensor value, Tensor? cum_seq_q, Tensor? cum_seq_k, SymInt max_q, SymInt max_k, float dropout_p, bool is_causal, bool return_debug_mask, *, float? scale=None, SymInt? window_size_left=None, SymInt? window_size_right=None, Tensor? seqused_k=None, Tensor? alibi_slopes=None) -> (Tensor output, Tensor softmax_logsumexp, Tensor philox_seed, Tensor philox_offset, Tensor debug_attn_mask)
    registered at /home/pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: CUDA
  previous kernel: registered at /home/pytorch/torch/csrc/autograd/generated/VariableType_0.cpp:18555
       new kernel: registered at /dev/null:241 (Triggered internally at /home/pytorch/aten/src/ATen/core/di

In [16]:
import torch

device = torch.device("cuda")

a = torch.randn(
    256,
    256,
    device=device,
    dtype=torch.float16,
)

b = torch.randn(
    256,
    256,
    device=device,
    dtype=torch.float16,
)

try:
    with torch.inference_mode():
        c = torch.mm(a, b)

    torch.cuda.synchronize()

    print("torch.mm 执行成功")
    print("输出形状：", tuple(c.shape))
    print("输出设备：", c.device)
    print("输出是否有限：", bool(torch.isfinite(c).all()))

except Exception as exc:
    print("torch.mm 仍然失败")
    print("错误类型：", type(exc).__name__)
    print("错误内容：", repr(exc))
    raise

torch.mm 执行成功
输出形状： (256, 256)
输出设备： cuda:0
输出是否有限： True


In [17]:
import torch

assert torch.cuda.is_available(), "当前 CUDA 不可用"

device_index = torch.cuda.current_device()
properties = torch.cuda.get_device_properties(device_index)

print("CUDA 设备编号：", device_index)
print("GPU 型号：", properties.name)
print("CUDA Compute Capability：", properties.major, properties.minor)
print(
    "总显存：",
    f"{properties.total_memory / 1024**3:.2f} GB",
)

shared_memory = getattr(
    properties,
    "shared_memory_per_block",
    None,
)

shared_memory_optin = getattr(
    properties,
    "shared_memory_per_block_optin",
    None,
)

print(
    "每个 block 默认共享内存：",
    shared_memory,
    "Bytes",
)

print(
    "每个 block opt-in 共享内存：",
    shared_memory_optin,
    "Bytes",
)

if shared_memory is not None:
    print(
        "每个 block 默认共享内存：",
        f"{shared_memory / 1024:.2f} KiB",
    )

if shared_memory_optin is not None:
    print(
        "每个 block opt-in 共享内存：",
        f"{shared_memory_optin / 1024:.2f} KiB",
    )

CUDA 设备编号： 0
GPU 型号： K100_AI
CUDA Compute Capability： 9 2
总显存： 63.98 GB
每个 block 默认共享内存： None Bytes
每个 block opt-in 共享内存： None Bytes


In [18]:
print("开始 FlagGems 推理……")

gems_output, gems_time, gems_peak_memory = run_one_step(seed=0)

print("\nFlagGems 推理完成。")
print("输出形状：", tuple(gems_output.shape))
print("运行时间：", f"{gems_time:.6f} 秒")
print("输出最小值：", gems_output.min().item())
print("输出最大值：", gems_output.max().item())
print("输出平均值：", gems_output.mean().item())
print("输出标准差：", gems_output.std().item())
print("全部为有限数值：", bool(torch.isfinite(gems_output).all()))

if gems_peak_memory is not None:
    print("峰值设备内存：", f"{gems_peak_memory:.3f} GB")

assert tuple(gems_output.shape) == tuple(baseline_output.shape), (
    "FlagGems 与原生 PyTorch 输出形状不一致。"
)

assert torch.isfinite(gems_output).all(), (
    "FlagGems 输出出现 NaN 或 Inf。"
)

absolute_difference = torch.abs(
    gems_output - baseline_output
)

max_absolute_error = absolute_difference.max().item()
mean_absolute_error = absolute_difference.mean().item()

baseline_norm = torch.linalg.vector_norm(
    baseline_output.reshape(-1)
).clamp_min(1e-12)

relative_l2_error = (
    torch.linalg.vector_norm(
        (gems_output - baseline_output).reshape(-1)
    )
    / baseline_norm
).item()

preliminary_allclose = torch.allclose(
    gems_output,
    baseline_output,
    rtol=1e-2,
    atol=1e-2,
)

print("\n" + "=" * 70)
print("FlagGems 与原生 PyTorch 数值对比")
print("=" * 70)
print("最大绝对误差：", max_absolute_error)
print("平均绝对误差：", mean_absolute_error)
print("相对 L2 误差：", relative_l2_error)
print("初步 allclose(rtol=1e-2, atol=1e-2)：", preliminary_allclose)

gems_residual_physical = dataset.unstandardize_t(
    gems_output,
    delta=6,
)

gems_next_state = (
    initial_state_physical + gems_residual_physical
)

gems_t2m_rmse = torch.sqrt(
    torch.mean(
        (
            gems_next_state[:, t2m_index]
            - true_next_state[:, t2m_index]
        ) ** 2
    )
).item()

print("\n2m_temperature 单样本 RMSE")
print("原生 PyTorch：", baseline_t2m_rmse)
print("FlagGems：", gems_t2m_rmse)
print(
    "二者 RMSE 差值：",
    abs(gems_t2m_rmse - baseline_t2m_rmse),
)

FUNCTIONAL_PASS = bool(
    torch.isfinite(gems_output).all()
    and tuple(gems_output.shape) == tuple(baseline_output.shape)
)

PRELIMINARY_NUMERICAL_PASS = bool(
    FUNCTIONAL_PASS and preliminary_allclose
)

print("\n功能运行是否通过：", FUNCTIONAL_PASS)
print("初步数值一致性是否通过：", PRELIMINARY_NUMERICAL_PASS)

开始 FlagGems 推理……

FlagGems 推理完成。
输出形状： (1, 69, 128, 256)
运行时间： 2.699235 秒
输出最小值： -19.373313903808594
输出最大值： 24.309856414794922
输出平均值： 0.0042643616907298565
输出标准差： 1.049100637435913
全部为有限数值： True
峰值设备内存： 1.457 GB

FlagGems 与原生 PyTorch 数值对比
最大绝对误差： 15.83292007446289
平均绝对误差： 0.2799477279186249
相对 L2 误差： 0.4169567823410034
初步 allclose(rtol=1e-2, atol=1e-2)： False

2m_temperature 单样本 RMSE
原生 PyTorch： 0.6926568150520325
FlagGems： 0.6502230763435364
二者 RMSE 差值： 0.042433738708496094

功能运行是否通过： True
初步数值一致性是否通过： False


In [19]:
import logging

# 强制刷新 Python 日志缓冲区
for logger_object in [
    logging.getLogger(),
    *[
        obj
        for obj in logging.Logger.manager.loggerDict.values()
        if isinstance(obj, logging.Logger)
    ],
]:
    for handler in logger_object.handlers:
        try:
            handler.flush()
        except Exception:
            pass

print("日志路径：", GEMS_LOG_PATH)
print("日志是否存在：", GEMS_LOG_PATH.exists())

assert GEMS_LOG_PATH.exists(), (
    "没有生成 gems_debug.log。\n"
    "请确认：\n"
    "1. FlagGems enable 单元在 FlagGems 推理之前执行；\n"
    "2. FlagGems 推理单元确实完成；\n"
    "3. path 指向当前有写权限的源码目录。"
)

log_size = GEMS_LOG_PATH.stat().st_size

print("日志大小：", log_size, "Bytes")

assert log_size > 0, (
    "gems_debug.log 存在但为空，说明没有记录到有效算子。"
)

log_text = GEMS_LOG_PATH.read_text(
    encoding="utf-8",
    errors="replace",
)

log_lines = log_text.splitlines()

print("日志行数：", len(log_lines))

print("\n日志前 80 行：")
print("-" * 70)
print("\n".join(log_lines[:80]))

if len(log_lines) > 80:
    print("\n日志最后 20 行：")
    print("-" * 70)
    print("\n".join(log_lines[-20:]))

print("\nFlagGems 日志检查通过。")

日志路径： /root/private_data/zc/swift-main/gems_debug.log
日志是否存在： True
日志大小： 1258 Bytes
日志行数： 23

日志前 80 行：
----------------------------------------------------------------------
[DEBUG] flag_gems.ops.randn.randn: GEMS RANDN
[DEBUG] flag_gems.ops.zeros.zero_: GEMS ZERO_
[DEBUG] flag_gems.ops.isfinite.isfinite: GEMS ISFINITE
[DEBUG] flag_gems.ops.copy.copy_: GEMS COPY_
[DEBUG] flag_gems.ops.normal.normal_: GEMS NORMAL_
[DEBUG] flag_gems.ops.zeros.zeros: GEMS ZEROS
[DEBUG] flag_gems.ops.cat.cat: GEMS CAT
[DEBUG] flag_gems.ops.mul.mul: GEMS MUL
[DEBUG] flag_gems.ops.repeat.repeat: GEMS REPEAT
[DEBUG] flag_gems.ops.addmm.addmm: GEMS ADDMM, [shape info]: [-, 8192, 1056, 564](batch, M, N, K), [A column-major]: False, [B column-major]: True, [bias column-major]: True
[DEBUG] flag_gems.ops.add.add: GEMS ADD
[DEBUG] flag_gems.ops.cos.cos: GEMS COS
[DEBUG] flag_gems.ops.sin.sin: GEMS SIN
[DEBUG] flag_gems.ops.flip.flip: GEMS FLIP
[DEBUG] flag_gems.ops.repeat_interleave.repeat_interleave_self_int: GE

In [20]:
def version_or_unknown(module_name, distribution_name=None):
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", None)
        if version is not None:
            return str(version)
    except Exception:
        pass

    try:
        return metadata.version(distribution_name or module_name)
    except Exception:
        return "未知"

device_description = str(DEVICE)

if DEVICE.type == "cuda" and torch.cuda.is_available():
    device_description = torch.cuda.get_device_name(DEVICE)

installed_text = (
    ", ".join(INSTALLED_THIS_RUN)
    if INSTALLED_THIS_RUN
    else "未额外安装运行依赖"
)

report_summary = f"""
# Swift / FlagGems 模型适配测试摘要

## 1. 模型信息

- 模型仓库：stockeh/swift-era5-1.4
- 验证变体：{MODEL_INFO["display_name"]}
- 配置目录：{MODEL_BASE}
- Checkpoint：{CHECKPOINT_NAME}
- 测试任务：ERA5 1.40625° 天气预报
- 测试规模：1 个样本、1 个随机成员、1 个 6 小时预测步
- 输入空间大小：{dataset.img_resolution}
- 输入条件形状：{tuple(condition.shape)}
- 模型输出形状：{tuple(gems_output.shape)}

## 2. 软件环境

- Python：{platform.python_version()}
- PyTorch：{torch.__version__}
- Triton：{version_or_unknown("triton")}
- FlagGems：{version_or_unknown("flag_gems", "flag-gems")}
- h5py：{version_or_unknown("h5py")}
- Hydra：{version_or_unknown("hydra", "hydra-core")}
- OmegaConf：{version_or_unknown("omegaconf")}
- 设备：{device_description}

## 3. 安装记录

- Swift 安装命令：
  {sys.executable} -m pip install --no-cache-dir --no-deps -e {ROOT}
- 本次补充安装的包：
  {installed_text}
- 未重新安装 torch、triton、flag_gems。

## 4. FlagGems 设置

- unused：
  batch_norm
  batch_norm_backward
- record：True
- once：True
- 日志：{GEMS_LOG_PATH}
- 日志大小：{log_size} Bytes
- 日志行数：{len(log_lines)}

## 5. 原生 PyTorch 结果

- 成功运行：True
- 输出全部为有限值：{bool(torch.isfinite(baseline_output).all())}
- 首次运行时间：{baseline_time:.6f} 秒
- 峰值设备内存：{baseline_peak_memory}
- 2m_temperature 单样本 RMSE：{baseline_t2m_rmse}

## 6. FlagGems 结果

- 成功运行：{FUNCTIONAL_PASS}
- 输出全部为有限值：{bool(torch.isfinite(gems_output).all())}
- 首次运行时间：{gems_time:.6f} 秒
- 峰值设备内存：{gems_peak_memory}
- 2m_temperature 单样本 RMSE：{gems_t2m_rmse}

## 7. 数值一致性

- 最大绝对误差：{max_absolute_error}
- 平均绝对误差：{mean_absolute_error}
- 相对 L2 误差：{relative_l2_error}
- allclose，rtol=1e-2，atol=1e-2：{preliminary_allclose}

## 8. 初步结论

- 功能适配：{"通过" if FUNCTIONAL_PASS else "未通过"}
- 初步数值一致性：{"通过" if PRELIMINARY_NUMERICAL_PASS else "需要进一步分析"}
- 本次属于单样本最小功能验证，不代表完整数据集科学精度或正式性能结果。
"""

print(report_summary)


# Swift / FlagGems 模型适配测试摘要

## 1. 模型信息

- 模型仓库：stockeh/swift-era5-1.4
- 验证变体：Swift
- 配置目录：weights/swift/020000
- Checkpoint：checkpoint-020000.pt
- 测试任务：ERA5 1.40625° 天气预报
- 测试规模：1 个样本、1 个随机成员、1 个 6 小时预测步
- 输入空间大小：(128, 256)
- 输入条件形状：(1, 72, 128, 256)
- 模型输出形状：(1, 69, 128, 256)

## 2. 软件环境

- Python：3.10.12
- PyTorch：2.4.1
- Triton：3.0.0
- FlagGems：5.0.0
- h5py：3.16.0
- Hydra：1.3.2
- OmegaConf：2.3.0
- 设备：K100_AI

## 3. 安装记录

- Swift 安装命令：
  /usr/bin/python3 -m pip install --no-cache-dir --no-deps -e /root/private_data/zc/swift-main
- 本次补充安装的包：
  未额外安装运行依赖
- 未重新安装 torch、triton、flag_gems。

## 4. FlagGems 设置

- unused：
  batch_norm
  batch_norm_backward
- record：True
- once：True
- 日志：/root/private_data/zc/swift-main/gems_debug.log
- 日志大小：1258 Bytes
- 日志行数：23

## 5. 原生 PyTorch 结果

- 成功运行：True
- 输出全部为有限值：True
- 首次运行时间：15.709348 秒
- 峰值设备内存：1.398695468902588
- 2m_temperature 单样本 RMSE：0.6926568150520325

## 6. FlagGems 结果

- 成功运行：True
- 输出全部为有限值：True
- 首次运行时间：2.699235 秒
- 峰值设备内存：1.45710611343